# DSORNet-v3.1 — Google Colab GPU

This notebook runs DSORNet-v3.1 on an NVIDIA Colab GPU using the same repository code used locally.

Before running, select **Runtime → Change runtime type → T4 GPU** or another available NVIDIA GPU.


## 1. Verify CUDA


In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not enabled. In Colab choose Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)


## 2. Clone DSOR and install Colab-safe dependencies

Colab already includes a CUDA-enabled PyTorch build. The Colab requirements file deliberately does not reinstall PyTorch.


In [ ]:
!rm -rf /content/DSOR
!git clone -q https://github.com/godsonj64/DSOR.git /content/DSOR
%cd /content/DSOR
!python -m pip install -q -r requirements-colab.txt


## 3. Run repository checks


In [ ]:
!pytest -q


## 4. Verify DSORNet executes on the GPU


In [ ]:
import torch
from dsorn_v31 import DSORNetV31Sequential

device = torch.device("cuda")
model = DSORNetV31Sequential(num_classes=10).to(device)
x = torch.randn(2, 3, 32, 32, device=device)

with torch.no_grad():
    logits = model(x)

print("Device:", next(model.parameters()).device)
print("Output shape:", tuple(logits.shape))
print("Finite:", bool(torch.isfinite(logits).all()))
print("Allocated GPU memory (MiB):", round(torch.cuda.memory_allocated() / 2**20, 2))


## 5. Configure a CIFAR training run


In [ ]:
DATASET = "cifar10"       # "cifar10" or "cifar100"
TRAIN_PER_CLASS = 100
TEST_PER_CLASS = 50
EPOCHS = 12
BATCH_SIZE = 128

print({
    "dataset": DATASET,
    "train_per_class": TRAIN_PER_CLASS,
    "test_per_class": TEST_PER_CLASS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
})


## 6. Train on CUDA


In [ ]:
import subprocess

cmd = [
    "python", "dsorn_v31.py",
    "--dataset", DATASET,
    "--data-dir", "/content/data",
    "--train-per-class", str(TRAIN_PER_CLASS),
    "--test-per-class", str(TEST_PER_CLASS),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--device", "cuda",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 7. Optional: checkpointed multi-seed experiment

This uses the cleaned experiment runner and writes a checkpoint plus `summary.json`.


In [ ]:
EXPERIMENT_DIR = "/content/cifar_results"

# A single seed is a convenient first run. Add more seeds for a stronger estimate.
experiment_cmd = [
    "python", "cifar_experiment.py",
    "--dataset", DATASET,
    "--data-dir", "/content/data",
    "--output-dir", EXPERIMENT_DIR,
    "--train-per-class", str(TRAIN_PER_CLASS),
    "--test-per-class", str(TEST_PER_CLASS),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--seeds", "20260929",
    "--device", "cuda",
]

# Uncomment to execute:
# subprocess.run(experiment_cmd, check=True)


## 8. Optional: copy saved results to Google Drive


In [ ]:
from pathlib import Path

# Uncomment after running the checkpointed experiment:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/DSOR
# !cp -r /content/cifar_results /content/drive/MyDrive/DSOR/


## Notes

- The architecture contains no convolution modules.
- CUDA uses the same PyTorch implementation as the local code; there is no separate Colab model fork.
- The small-subset CIFAR commands are research pilots, not full-dataset benchmark claims.
- Colab runtimes are ephemeral; persist experiment outputs if you need them after the session.
